# Import dos dados e bibliotecas

In [2]:
pip install gensim --pre

  Using cached gensim-4.4.0.tar.gz (23.3 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached smart_open-8.0.1-py3-none-any.whl.metadata (24 kB)
Using cached smart_open-8.0.1-py3-none-any.whl (73 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for gensim (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [858 lines of output]
      /tmp/pip-build-env-qi29k30m/overlay/lib/python3.14/site-packages/setuptools/_distutils/dist.py:288: UserWarning: Unknown distribution option: 'test_suite'
        warnings.warn(msg)
      /tmp/pip-build-env-qi29k30m/overlay/lib/python3.14/site-packages/setuptools/_distutils/dist.py:288: UserWarning: Unknown distribution option: 'tests_require'
        warnings.warn(msg)
      running bdist_wheel
      running build
      running build_py
      creating build/lib.linux-x86_64-cpython-314/gensim
      copying gensim/__init__.py ->

In [2]:
import pandas as pd
import numpy as np

from gensim.models import Word2Vec

In [4]:
df_caminhao = pd.read_parquet('/content/df_caminhao_limpo.parquet', engine='pyarrow')

# Extração das sequências de erros

In [5]:
import pandas as pd
import unicodedata
import re

def limpar_texto(texto):
    if not isinstance(texto, str):
        return ""

    # 1. Remove acentos e caracteres como 'ç' (Normalização Unicode)
    # Ex: 'queda_pressão' -> 'queda_pressao'
    texto_normalizado = unicodedata.normalize('NFKD', texto)
    texto_limpo = "".join([c for c in texto_normalizado if not unicodedata.combining(c)])

    # 2. Converte para minúsculas (ou maiúsculas, o que você preferir padronizar)
    texto_limpo = texto_limpo.lower()

    # 3. Substitui hífens soltos, barras e caracteres especiais por underscore '_'
    # Mantém apenas letras, números e underscores
    texto_limpo = re.sub(r'[^a-z0-9_]', '_', texto_limpo)

    # 4. Remove múltiplos underscores seguidos (ex: 'mc_-_queda' vira 'mc___queda' e depois 'mc_queda')
    texto_limpo = re.sub(r'_+', '_', texto_limpo)

    # 5. Remove underscores sobressalentes no início ou fim do texto
    texto_limpo = texto_limpo.strip('_')

    return texto_limpo

# =====================================================================
# APLICANDO A LIMPEZA NO DATASET
# =====================================================================

# Criamos cópias limpas das colunas originais
df_caminhao['Alarme_Limpo'] = df_caminhao['Alarme'].apply(limpar_texto)
df_caminhao['Situacao_Limpa'] = df_caminhao['Situacao_Operacional'].apply(limpar_texto)

# Criamos o token composto contextualizado
df_caminhao['Alarme_Token'] = df_caminhao['Alarme_Limpo'] #+ '_' + df_caminhao['Situacao_Limpa']
df_caminhao['Alarme_Token']

,Alarme_Token
0,mc_queda_pressao_do_oleo_motor
1,mc_queda_pressao_do_oleo_motor
2,oem_interface_timeout
3,oem_interface_normal
4,mc_queda_pressao_do_oleo_motor
...,...
840653,oem_interface_normal
840654,parking_brake_active
840655,oem_interface_timeout
840656,oem_interface_normal


In [6]:
# =====================================================================
# 1. CONFIGURAÇÃO DO LIMITE TEMPORAL
# =====================================================================
# Definimos o limite de tempo para quebrar a sequência (ex: 6 horas)
LIMITE_HORAS = 12
limite_tempo = pd.Timedelta(hours=LIMITE_HORAS)

# 2. ORDENAÇÃO CRUCIAL
df_caminhao = df_caminhao.sort_values(by=['TAG', 'Data_Evento']).reset_index(drop=True)

# =====================================================================
# 3. CÁLCULO DOS INTERVALOS E QUEBRA DE SESSÕES
# =====================================================================
# Calcula a diferença de tempo entre o log atual e o anterior do MESMO veículo
df_caminhao['Diferenca_Tempo'] = df_caminhao.groupby('TAG')['Data_Evento'].diff()

# Marca True sempre que a diferença for maior que o limite (ou se for o primeiro registro do veículo, que vem como NaT)
df_caminhao['Nova_Sessao_Trigger'] = (df_caminhao['Diferenca_Tempo'] > limite_tempo) | df_caminhao['Diferenca_Tempo'].isna()

# O cumsum() cria um ID incremental toda vez que encontra um True.
# Isso gera IDs de "Sessões de Trabalho" contínuas para cada caminhão.
df_caminhao['ID_Sessao'] = df_caminhao.groupby('TAG')['Nova_Sessao_Trigger'].cumsum()

# =====================================================================
# 4. CRIAÇÃO DAS SEQUÊNCIAS CONTEXTUAIS POR SESSÃO
# =====================================================================
# Agora agrupamos por TAG E pelo ID da Sessão de trabalho
sequencias_finais = (
    df_caminhao
    .groupby(['TAG', 'ID_Sessao'])['Alarme_Token']
    .apply(list)
    .tolist()
)

# Filtramos para remover "frases" que ficaram vazias ou com apenas 1 alarme
# (Word2Vec precisa de pelo menos 2 palavras para aprender relações de contexto)
sequencias_filtradas = [seq for seq in sequencias_finais if len(seq) >= 2]

In [7]:
sequencias_filtradas[0]

['mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'oem_interface_timeout',
 'oem_interface_normal',
 'mc_queda_pressao_do_oleo_motor',
 'payload_overload_active',
 'left_rear_brake_temperature_active',
 'right_rear_brake_temperature_active',
 'payload_overload_active',
 'payload_overload_dump_active',
 'oem_interface_timeout',
 'oem_interface_normal',
 'oem_interface_timeout',
 'oem_interface_normal',
 'oem_interface_timeout',
 'oem_interface_normal',
 'oem_interface_timeout',
 'oem_interface_normal',
 'oem_interface_timeout',
 'oem_interface_normal',
 'op_baixo_nivel_de_combustivel_20',
 'mc_queda_pressao_do_oleo_motor',
 'oem_interface_timeout',
 'oem_interface_normal',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao_do_oleo_motor',
 'mc_queda_pressao

In [8]:
model_w2v = Word2Vec(
    sentences=sequencias_filtradas,
    vector_size=32,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=25,
    sample=1e-4
)
print("✅ Word2Vec Contextual treinado com sucesso!")

✅ Word2Vec Contextual treinado com sucesso!


In [9]:
# Escolha um alarme que você sabe que é crítico na sua base
alarme_teste = "right_turbo_in_pressure_active"

if alarme_teste in model_w2v.wv:
    print(f"Alarmes mais correlacionados com '{alarme_teste}':")
    for alarme, similaridade in model_w2v.wv.most_similar(alarme_teste, topn=15):
        print(f" - {alarme}: {similaridade:.4f}")
else:
    print(f"O alarme '{alarme_teste}' não foi encontrado no vocabulário.")

Alarmes mais correlacionados com 'right_turbo_in_pressure_active':
 - left_turbocharger_inlet_pressure_active: 0.8745
 - atmos_pressure_active: 0.8384
 - altitude_derate_active: 0.7810
 - right_turbocharger_inlet_pressure_active: 0.7768
 - front_aftercooler_temperature_active: 0.6587
 - engine_coolant_temperature_active: 0.5569
 - engine_cool_temperature_active: 0.5561
 - 5v_supply_active: 0.5546
 - boost_pressure_active: 0.5371
 - unfiltered_oil_pressure_active: 0.5113
 - low_oil_pressure_active: 0.5041
 - parkbrake_active: 0.4859
 - engine_oil_filter_active: 0.4730
 - injector_cylinder_12_active: 0.4568
 - mc_dif_pressao_de_admissao_do_turbo_3kpa: 0.4478


In [10]:
model_w2v.wv.index_to_key

['final_drive_bypass_solenoid_active',
 'engine_coolant_level_active',
 '1st_tire_tag_timeout',
 'oem_interface_timeout',
 'oem_interface_normal',
 'mc_temperatura_da_direcao_95c',
 'payload_overload_active',
 'steering_oil_temperature_active',
 'ma_baixa_voltagem_com_equip_desligado_24v',
 'shutdown_inputs_active',
 'differential_lube_pressure_active',
 'transmission_output_speed2_active',
 'start_motor_relay_active',
 'ma_dif_temp_exaust_60_oc',
 'engine_prelube_active',
 'brake_servomechanism_solenoid_active',
 'ambient_air_temperature_active',
 'mc_queda_pressao_do_oleo_motor_280kpa',
 'payload_overload_dump_active',
 'ma_diferenca_de_temperatura_de_freio_te_td_25_c',
 'tcs_falha_sensor_de_rotacao_da_roda_tras_esq',
 'mc_baixa_pressao_de_ar_de_freio_90psi',
 'differential_filter_switch_active',
 'mi_filtro_oleo_motor_obstruido_69kpa',
 'air_filter_active',
 'mc_queda_pressao_do_oleo_motor',
 'right_front_brake_temperature_active',
 'ma_diferenca_de_temperatura_de_freio_de_dd_25_c',

# Extração de Embeddings e construção da média EWMA

In [11]:
# =====================================================================
# 1. EXTRAÇÃO DOS EMBEDDINGS (VETORES DE CARACTERÍSTICAS)
# =====================================================================
print("Extraindo vetores do Word2Vec...")
tamanho_vetor = model_w2v.vector_size
colunas_vetor = [f'Dim_{i}' for i in range(tamanho_vetor)]

def obter_vetor(alarme):
    if alarme in model_w2v.wv:
        return model_w2v.wv[alarme]
    else:
        return np.zeros(tamanho_vetor)

# Extrai os vetores e os transforma em colunas do DataFrame original
vetores = df_caminhao['Alarme_Limpo'].apply(obter_vetor)
df_vetores = pd.DataFrame(vetores.tolist(), index=df_caminhao.index, columns=colunas_vetor)
df_caminhao_vetorizado = pd.concat([df_caminhao, df_vetores], axis=1)

Extraindo vetores do Word2Vec...


In [12]:
# =====================================================================
# PREPARAÇÃO BLINDADA PARA OPERAÇÕES TEMPORAIS (SIMPLIFICADA E LIMPA)
# =====================================================================
print("Preparando dados multirresolução...")

# 1. ORDENAÇÃO CRUCIAL: Por Veículo e depois pelo Tempo
df_enriquecido = df_caminhao_vetorizado.sort_values(by=['TAG', 'Data_Evento']).reset_index(drop=True)

# 2. Criamos uma "Cópia Espelho" apenas para fazer os cálculos de tempo
# Garantimos que o índice é datetime (obrigatório para o decaimento temporal)
df_calculo = df_enriquecido.set_index(pd.to_datetime(df_enriquecido['Data_Evento']))

# ==========================================
# A. MÚLTIPLAS MEIAS-VIDAS DOS EMBEDDINGS (FÍSICA)
# ==========================================
meias_vidas = {'2h': '2 hours', '12h': '12 hours', '72h': '72 hours'}

for nome, tempo in meias_vidas.items():
    print(f"Calculando EWMA para meia-vida: {nome}...")

    # Ao colocar a data como índice, o Pandas entende automaticamente o 'halflife'
    # Adicionamos 'times=df_calculo.index' para explicitar a coluna de tempo
    resultado_ewm = df_calculo.groupby('TAG')[colunas_vetor].ewm(halflife=tempo, times=df_calculo.index).mean()

    # Extraímos o array bruto (.values) garantindo que a ordem bata perfeitamente
    novas_colunas = [f'EWMA_{nome}_{col}' for col in colunas_vetor]
    df_enriquecido[novas_colunas] = resultado_ewm.values

# ==========================================
# B. LIMPEZA DOS VETORES INSTANTÂNEOS (RUÍDO)
# ==========================================
print("Removendo vetores instantâneos originais para reduzir ruído...")
df_enriquecido.drop(columns=colunas_vetor, inplace=True)

print(f"✅ Base pronta e enxuta! Total de colunas agora: {df_enriquecido.shape[1]}")

Preparando dados multirresolução...
Calculando EWMA para meia-vida: 2h...
Calculando EWMA para meia-vida: 12h...
Calculando EWMA para meia-vida: 72h...


/tmp/ipykernel_1656/967122411.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_enriquecido[novas_colunas] = resultado_ewm.values
/tmp/ipykernel_1656/967122411.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_enriquecido[novas_colunas] = resultado_ewm.values
/tmp/ipykernel_1656/967122411.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a

Removendo vetores instantâneos originais para reduzir ruído...
✅ Base pronta e enxuta! Total de colunas agora: 119


In [13]:
df_enriquecido.to_parquet('df_caminhao_modelagem.parquet', index=False)